# Build Review Summarie

select informative Steam reviews per game and compress them into one review_summary text block per game

In [1]:
import sys
print(sys.executable)

c:\Users\heloi\OneDrive - KU Leuven\EMOS\adv analytics BD\assignment 3\.venv\Scripts\python.exe


In [2]:
from __future__ import annotations

import re
import sqlite3
from pathlib import Path

import pandas as pd

pd.set_option("display.max_colwidth", 120)

DB_PATH = Path("steam_games_reviews_25.sqlite")
OUTPUT_PATH = Path("review_summaries.csv")
MIN_REVIEW_LENGTH = 20
MAX_REVIEWS_PER_GAME = 5

assert DB_PATH.exists(), f"Database not found: {DB_PATH.resolve()}"
print(DB_PATH.resolve())

C:\Users\heloi\OneDrive - KU Leuven\EMOS\adv analytics BD\assignment 3\steam_games_reviews_25.sqlite


## Load raw candidate reviews

only load english reviews

In [3]:
CANDIDATE_POOL_PER_SENTIMENT = 10
MAX_GAMES = 2000  # Set to None for the full dataset once you are happy with the quality.


def load_review_candidates(
    db_path: Path,
    min_review_length: int = 20,
    candidate_pool_per_sentiment: int = 20,
    max_games: int | None = None,
) -> pd.DataFrame:
    conn = sqlite3.connect(db_path)

    if max_games is None:
        query = """
        WITH filtered_reviews AS (
            SELECT
                r.appid,
                r.review,
                r.voted_up,
                r.votes_up,
                CAST(r.weighted_vote_score AS REAL) AS weighted_vote_score
            FROM reviews r
            WHERE r.language = 'english'
              AND r.review IS NOT NULL
              AND TRIM(r.review) != ''
              AND LENGTH(TRIM(r.review)) >= ?
        ),
        ranked_reviews AS (
            SELECT
                fr.*,
                ROW_NUMBER() OVER (
                    PARTITION BY fr.appid, fr.voted_up
                    ORDER BY
                        fr.votes_up DESC,
                        fr.weighted_vote_score DESC,
                        LENGTH(fr.review) DESC
                ) AS rn
            FROM filtered_reviews fr
        )
        SELECT
            appid,
            review,
            voted_up,
            votes_up,
            weighted_vote_score
        FROM ranked_reviews
        WHERE rn <= ?
        """
        params = (min_review_length, candidate_pool_per_sentiment)
    else:
        query = """
        WITH limited_games AS (
            SELECT appid
            FROM games
            ORDER BY appid
            LIMIT ?
        ),
        filtered_reviews AS (
            SELECT
                r.appid,
                r.review,
                r.voted_up,
                r.votes_up,
                CAST(r.weighted_vote_score AS REAL) AS weighted_vote_score
            FROM reviews r
            JOIN limited_games lg ON lg.appid = r.appid
            WHERE r.language = 'english'
              AND r.review IS NOT NULL
              AND TRIM(r.review) != ''
              AND LENGTH(TRIM(r.review)) >= ?
        ),
        ranked_reviews AS (
            SELECT
                fr.*,
                ROW_NUMBER() OVER (
                    PARTITION BY fr.appid, fr.voted_up
                    ORDER BY
                        fr.votes_up DESC,
                        fr.weighted_vote_score DESC,
                        LENGTH(fr.review) DESC
                ) AS rn
            FROM filtered_reviews fr
        )
        SELECT
            appid,
            review,
            voted_up,
            votes_up,
            weighted_vote_score
        FROM ranked_reviews
        WHERE rn <= ?
        """
        params = (max_games, min_review_length, candidate_pool_per_sentiment)

    try:
        return pd.read_sql_query(query, conn, params=params)
    finally:
        conn.close()


candidate_reviews_df = load_review_candidates(
    DB_PATH,
    min_review_length=MIN_REVIEW_LENGTH,
    candidate_pool_per_sentiment=CANDIDATE_POOL_PER_SENTIMENT,
    max_games=MAX_GAMES,
)
print(f"Loaded {len(candidate_reviews_df):,} candidate review rows.")
candidate_reviews_df.head()

Loaded 39,475 candidate review rows.


,appid,review,voted_up,votes_up,weighted_vote_score
0,10,bunch of fake servers w fake ping fake player count... maybe even a virus,0,8,0.570079
1,10,Absolutely Not Recommended\r\n\r\nI genuinely cannot stand Counter-Strike (2000). I don’t care how “legendary” peopl...,0,5,0.490681
2,10,do not give money to valve you do not own your games you own a liscense\ndo not buy \nthe steam deck\nsteam machine ...,0,4,0.500430
3,10,why dafuk would they remove add_bots command?,0,1,0.514787
4,10,"Once you look past the nostalgia, there isn't much of a reason to return to this title. The player base is dwindling...",0,1,0.509660


## Review cleaning and filtering helpers

In [4]:
def normalize_review_text(text: str) -> str:
    text = text.replace("\r\n", "\n").replace("\r", "\n") #standardise line breaks => all to \n
    text = re.sub(r"\n{3,}", "\n\n", text) #avoid multiple blank lines
    text = re.sub(r"[ \t]{2,}", " ", text) #repeated spaces/tabs => single space
    return text.strip() #remove leading/trailing whitespace


def is_informative_review(text: str) -> bool:
    text = normalize_review_text(text)
    if not text:
        return False #reject empty reviews after normalisation

    alpha_chars = sum(ch.isalpha() for ch in text)
    total_chars = len(text)
    alpha_ratio = alpha_chars / max(total_chars, 1) #how much of review is letters
    newline_ratio = text.count("\n") / max(total_chars, 1)
    unique_words = set(re.findall(r"[a-zA-Z]{2,}", text.lower())) #keep set of unique lowered words => vocab variety

    if alpha_ratio < 0.50: #no fewer than 50% of chars should be letters
        return False
    if newline_ratio > 0.1: #no more than 10% of chars should be newlines (avoid ascii art, or other sus formatting) 
        return False
    if len(unique_words) < 10: #avoid very basic reviews
        return False

    return True


def score_review(row: pd.Series) -> float:
    text = normalize_review_text(str(row["review"])) #clean it
    text_len = len(text)
    sentence_count = max(1, text.count(".") + text.count("!") + text.count("?"))
    alpha_chars = sum(ch.isalpha() for ch in text)
    alpha_ratio = alpha_chars / max(text_len, 1)

    score = 0.0
    score += float(row["votes_up"]) * 1.0 #other users found this review helpful
    score += float(row["weighted_vote_score"]) * 20.0 #the weighted score from stream, times 20 so it's noticeable
    score += min(text_len, 800) / 80.0 #reward longer reviews, but cap at 800 chars (10 points max)
    score += min(sentence_count, 6) * 2.0 #reward more sentences, but cap at 6 (12 points max)
    score += alpha_ratio * 10.0 #reward reviews with higher proportion of letters
    return score

In [5]:
def is_mostly_latin_text(text: str, min_ratio: float = 0.9) -> bool:
    letters = [ch for ch in text if ch.isalpha()]
    if not letters:
        return False

    latin_letters = [ch for ch in letters if ("A" <= ch <= "Z") or ("a" <= ch <= "z")]
    return (len(latin_letters) / len(letters)) >= min_ratio

## Apply filters and score reviews

In [6]:
filtered_reviews_df = candidate_reviews_df.copy()
filtered_reviews_df["review"] = filtered_reviews_df["review"].astype(str).apply(normalize_review_text)
filtered_reviews_df = filtered_reviews_df[filtered_reviews_df["review"].apply(is_informative_review)].copy()
filtered_reviews_df = filtered_reviews_df[
    filtered_reviews_df["review"].apply(is_mostly_latin_text)
].copy()
filtered_reviews_df["review_score"] = filtered_reviews_df.apply(score_review, axis=1)

print(f"Kept {len(filtered_reviews_df):,} informative reviews after filtering.")
filtered_reviews_df.head()

Kept 35,679 informative reviews after filtering.


,appid,review,voted_up,votes_up,weighted_vote_score,review_score
0,10,bunch of fake servers w fake ping fake player count... maybe even a virus,0,8,0.570079,34.122294
1,10,Absolutely Not Recommended\n\nI genuinely cannot stand Counter-Strike (2000). I don’t care how “legendary” people sa...,0,5,0.490681,44.705924
2,10,do not give money to valve you do not own your games you own a liscense\ndo not buy \nthe steam deck\nsteam machine ...,0,4,0.500430,26.250632
4,10,"Once you look past the nostalgia, there isn't much of a reason to return to this title. The player base is dwindling...",0,1,0.509660,29.568205
5,10,"Server Browser is filled with 255/255 fake servers, known to infect PC's with viruses.\nHiding full servers helps, b...",0,1,0.489691,28.455657


## Select a balanced set of reviews per game

keep up to 5 reviews per game, if possible 3 positive 2 negative (based on the steam "vote_up" label, assigned by the reviewer)

In [7]:
def select_balanced_reviews(reviews_df: pd.DataFrame, max_reviews_per_game: int = 5) -> pd.DataFrame:
    positive_quota = max_reviews_per_game - (max_reviews_per_game // 2)
    negative_quota = max_reviews_per_game // 2
    selected_groups = []

    for _, group in reviews_df.groupby("appid", sort=True): #loop over reviews game by game
        positives = (
            group[group["voted_up"] == 1]
            .sort_values("review_score", ascending=False)
            .head(positive_quota)
        )
        negatives = (
            group[group["voted_up"] == 0]
            .sort_values("review_score", ascending=False)
            .head(negative_quota)
        )

        selected = pd.concat([positives, negatives], ignore_index=False)

        if len(selected) < max_reviews_per_game:
            remaining = (
                group.drop(index=selected.index, errors="ignore")
                .sort_values("review_score", ascending=False)
                .head(max_reviews_per_game - len(selected))
            )
            selected = pd.concat([selected, remaining], ignore_index=False) #fill remaining slots with highest scored reviews regardless of sentiment

        if not selected.empty:
            selected = selected.sort_values(
                ["voted_up", "review_score"],
                ascending=[False, False],
            ).head(max_reviews_per_game) #present strongest positive evidence first
            selected_groups.append(selected)

    if not selected_groups:
        return reviews_df.iloc[0:0].copy()

    return pd.concat(selected_groups, ignore_index=True)


selected_reviews_df = select_balanced_reviews(filtered_reviews_df, max_reviews_per_game=MAX_REVIEWS_PER_GAME)
print(f"Selected {len(selected_reviews_df):,} reviews after balancing.")
selected_reviews_df.head()

Selected 9,966 reviews after balancing.


,appid,review,voted_up,votes_up,weighted_vote_score,review_score
0,10,---{ Graphics }---\n☐ You forget what reality is\n☐ Beautiful\n☐ Good\n☐ Decent\n☑ Bad\n☐ Don‘t look too long at it\...,1,100,0.862660,137.505050
1,10,"Counter-Strike is where it all started. Simple, raw, and purely skill-based. No fancy mechanics, no distractions — j...",1,53,0.754579,94.315774
2,10,"A classic. I'm in the younger generation of players for this game, my dad introduced me to it when I was five. Almos...",1,23,0.711582,65.012882
3,10,Absolutely Not Recommended\n\nI genuinely cannot stand Counter-Strike (2000). I don’t care how “legendary” people sa...,0,5,0.490681,44.705924
4,10,bunch of fake servers w fake ping fake player count... maybe even a virus,0,8,0.570079,34.122294


## Build one `review_summary` per game

selected reviews per game => one text block per game

so later we can combine metadata & reviews at game level

In [8]:
def build_review_summary(reviews_df: pd.DataFrame) -> pd.DataFrame:
    def label_review(row: pd.Series) -> str:
        sentiment = "positive" if row["voted_up"] == 1 else "negative"
        return f"[{sentiment}, helpful_votes={row['votes_up']}] {row['review']}"

    labeled = reviews_df.copy()
    labeled["labeled_review"] = labeled.apply(label_review, axis=1)

    summary_df = (
        labeled.groupby("appid")["labeled_review"]
        .apply(lambda rows: "\n".join(rows))
        .reset_index(name="review_summary")
    )
    return summary_df
#format selected reviews into summaries per game, with helpful votes and sentiment labels
#give the text more context for LLM

review_summaries_df = build_review_summary(selected_reviews_df)
print(f"Built {len(review_summaries_df):,} game summaries.")
review_summaries_df.head(10)

Built 1,994 game summaries.


,appid,review_summary
0,10,"[positive, helpful_votes=100] ---{ Graphics }---\n☐ You forget what reality is\n☐ Beautiful\n☐ Good\n☐ Decent\n☑ Bad..."
1,20,"[positive, helpful_votes=82] Since no one on TF community will read this review, I wish to the community, that inclu..."
2,30,"[positive, helpful_votes=88] I am 32 years old.\n\nMy ex-wife and I have a daughter together, and we adopted our son..."
3,40,"[positive, helpful_votes=84] If Deathmatch Classic has a million fans, I am one of them.\nIf Deathmatch Classic has ..."
4,50,"[positive, helpful_votes=9] Half-Life: Opposing Force introduces players to Corporal Adrian Shephard, a member of th..."
5,60,"[positive, helpful_votes=105] Ricochete Community (if exists) and valve games community. Since no One Will read this..."
6,70,"[positive, helpful_votes=2] Just an absolute banger of a game. Spent so much of my childhood on this thing, and it h..."
7,80,"[positive, helpful_votes=32] A single player campaign based game, still better than playing with cheaters on CS2.\nE..."
8,130,"[positive, helpful_votes=50] Half Life community. I know no one will read this review but I want to say something re..."
9,220,"[positive, helpful_votes=5] This is a deeply nostalgic game for me, being a 90s kid and growing up playing mostly sh..."


## Inspect a few examples

In [9]:
pd.set_option("display.max_colwidth", None)
review_summaries_df.head(10)

appid  \
0     10   
1     20   
2     30   
3     40   
4     50   
5     60   
6     70   
7     80   
8    130   
9    220   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              

## Save to CSV

In [10]:
review_summaries_df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved to: {OUTPUT_PATH.resolve()}")

Saved to: C:\Users\heloi\OneDrive - KU Leuven\EMOS\adv analytics BD\assignment 3\review_summaries.csv
